# xlingual-drift — Colab runner

Colab-side entrypoint (Section 15.8 / Appendix D of the project doc). Use this notebook
only for steps that actually need the GPU — open-weight model inference, or the optional
SFT ablation. Everything else (writing/debugging the environment, the verifier, stats,
API-model runs) should happen locally instead; this notebook exists so those two GPU steps
have somewhere to run on a single A100 (80GB VRAM, ~150GB system RAM).

**Workflow:** clone from GitHub every session (don't sync code through Drive) → mount
Drive only for bulky run logs / model weight caches → run → commit condensed results back
to GitHub before the session ends.

In [ ]:
# 1. Confirm the GPU we actually got.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone (or pull) the repo. Replace with your actual GitHub URL once it exists.
REPO_URL = "https://github.com/khalerishabh/xlingual-drift.git"

import os
if not os.path.exists("xlingual-drift"):
    !git clone {REPO_URL}
else:
    !cd xlingual-drift && git pull
%cd xlingual-drift

In [ ]:
# 3. Install dependencies. Uncomment the GPU-inference libraries you need
#    (transformers/accelerate/bitsandbytes or vllm) once local_policy.py is wired up --
#    they're deliberately not in requirements.txt yet, since they're Colab-only.
!pip install -q -r requirements.txt
# !pip install -q transformers accelerate bitsandbytes
# !pip install -q vllm

In [ ]:
# 4. Mount Drive -- for bulky, disposable artifacts ONLY (raw run logs, model weight
#    cache). Code stays in the git clone above, never in Drive.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RUNS_DIR = "/content/drive/MyDrive/xlingual-drift-runs"
import os
os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)

# Point the repo's runs/ output at Drive instead of the ephemeral Colab disk, so a
# disconnect doesn't lose a long run. Symlinking is simplest and keeps run_grid.py
# unmodified.
!rm -rf runs && ln -s {DRIVE_RUNS_DIR} runs

In [ ]:
# 5. API keys via Colab Secrets (the key icon in the left sidebar), never pasted in a cell.
from google.colab import userdata
import os
for key in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass  # fine if a given run doesn't need that provider

## Smoke test (proves the clone + install worked, no GPU needed)

In [ ]:
!python -m eval.run_grid --config configs/smoke_test.yaml

## Real GPU run (once agent/policies/local_policy.py is implemented)

Point a config's `runs[].policy` at `local` and it will load an open-weight model here.
Keep the model at or below ~32B (4-bit quantized if needed) — anything bigger doesn't fit
a single 80GB A100, which is why the Section 8.7 shortlist was trimmed (no Llama 3.1 70B,
no Qwen3-Next-80B-A3B, no Qwen3-VL-235B-A22B).

In [ ]:
# !python -m eval.run_grid --config configs/<your_gpu_config>.yaml

## Before ending the session

Commit the CONDENSED results (a metrics summary, not the raw jsonl logs -- those stay on
Drive) back to GitHub, so the run's headline numbers are versioned even though the full
logs aren't.

In [ ]:
# Example -- adapt once eval/metrics.py grows a real summary-export step.
# !git add configs/ results/summary.json
# !git commit -m "Colab run: <describe it>"
# !git push